# 01. Testing Ollama with LiteLLM and LangGraph

This notebook demonstrates how to:
1. Initialize and test the local Ollama LLM using LiteLLM.
2. Bind dynamic MCP tools from the Qdrant and OpenProject MCP servers.
3. Define custom extra python tools.
4. Construct a LangGraph-based ReAct agent that integrates LiteLLM for tool calling.
5. Implement a robust Human-in-the-Loop (HITL) gate for write operations.

In [1]:
import sys
from pathlib import Path

# Locate and append project root so src.ragforge is importable
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

project_root = get_project_root().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/sunnyraj/code_files/git_repos/RagForge


In [2]:
import litellm
from src.ragforge.config import OLLAMA_URL, DEFAULT_LLM_MODEL

print(f"Ollama Endpoint: {OLLAMA_URL}")
print(f"Model Name     : {DEFAULT_LLM_MODEL}")

# Check basic connection
response = litellm.completion(
    model=f"ollama/{DEFAULT_LLM_MODEL}",
    messages=[{"role": "user", "content": "Explain what a ReAct agent is in 2 sentences."}],
    api_base=OLLAMA_URL
)

print("\nResponse from LiteLLM:")
print(response.choices[0].message.content)

Ollama Endpoint: http://localhost:11434
Model Name     : gemma4:e4b

Response from LiteLLM:
A ReAct agent is an advanced AI framework that enables Large Language Models to solve complex tasks by combining internal reasoning with external tool use. It operates in a loop of "Thought," where it plans its next step, followed by an "Action" taken using tools, and finally an "Observation" of the result to guide subsequent decisions until the goal is reached.


In [3]:
# Demo LiteLLM tool calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name, e.g., San Francisco"}
                },
                "required": ["location"]
            }
        }
    }
]

response = litellm.completion(
    model=f"ollama/{DEFAULT_LLM_MODEL}",
    messages=[{"role": "user", "content": "What is the weather in Paris?"}],
    tools=tools,
    api_base=OLLAMA_URL
)

print("Tool Call generated by LLM:")
print(response.choices[0].message.tool_calls)

Tool Call generated by LLM:
[ChatCompletionMessageToolCall(function=Function(arguments='{"location": "Paris"}', name='get_weather'), id='call_c3f80983-88bb-4bb5-8ae2-0b2671759d3c', type='function')]


## 2. Load MCP and Custom Tools

Next, let's load dynamic MCP tools using the project's stdio client parameters for Qdrant and OpenProject, and define a few custom python tools.

In [4]:
import asyncio
from src.ragforge.agents.rag.agent import build_mcp_tools
from langchain_core.tools import tool

# 1. Load MCP tools
raw_mcp_tools, tool_server_map = await build_mcp_tools()
print(f"Loaded {len(raw_mcp_tools)} MCP tools.")

# 2. Define custom extra python tools
@tool
def calculate_math(expression: str) -> str:
    """Evaluate a mathematical expression. Input should be a valid math string like '2 + 2 * 10'."""
    try:
        val = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {val}"
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

@tool
def get_system_time() -> str:
    """Retrieve the current local system time."""
    from datetime import datetime
    return f"Current Local Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

# Wrap custom tools to match agent raw tools format
custom_tools = [
    {
        "name": "calculate_math",
        "description": calculate_math.description,
        "schema": calculate_math.args_schema.schema() if calculate_math.args_schema else {"type": "object", "properties": {"expression": {"type": "string"}}},
        "server": "custom",
        "callable": calculate_math
    },
    {
        "name": "get_system_time",
        "description": get_system_time.description,
        "schema": get_system_time.args_schema.schema() if get_system_time.args_schema else {"type": "object", "properties": {}},
        "server": "custom",
        "callable": get_system_time
    }
]

# Merge tools
all_tools = raw_mcp_tools + [{"name": t["name"], "description": t["description"], "schema": t["schema"], "server": t["server"]} for t in custom_tools]
tool_server_map.update({t["name"]: t["server"] for t in custom_tools})
custom_callable_map = {t["name"]: t["callable"] for t in custom_tools}

print(f"Total merged tools: {len(all_tools)}")

Loaded 10 MCP tools.
Total merged tools: 12


/var/folders/38/sknm76sx0pn_hk94zlgzh5v80000gn/T/ipykernel_98901/1059195123.py:30: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  "schema": calculate_math.args_schema.schema() if calculate_math.args_schema else {"type": "object", "properties": {"expression": {"type": "string"}}},
/var/folders/38/sknm76sx0pn_hk94zlgzh5v80000gn/T/ipykernel_98901/1059195123.py:37: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  "schema": get_system_time.args_schema.schema() if get_system_time.args_schema else {"type": "object", "properties": {}},


## 3. LangGraph Agent with LiteLLM & HITL

We will construct a LangGraph ReAct agent that:
1. Uses `litellm` inside its `agent` node to run completions and structure tool calls.
2. Implements a `check_hitl` node to trap write tools (like `upsert_documents` or `create_project_task`).
3. Interrupts execution at `hitl_pause` node, allowing human approval prior to executing tools.

In [5]:
import uuid
from typing import Literal, Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from src.ragforge.agents.rag.agent import execute_mcp_tool, WRITE_TOOLS

class AgentState(TypedDict):
    messages: list[dict]
    pending_tool_call: dict | None
    hitl_approved: bool | None

checkpointer = MemorySaver()

In [6]:
async def agent_node(state: AgentState):
    """Reasoning node that calls Ollama via LiteLLM."""
    messages = state["messages"]
    
    tools_payload = [
        {
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t["description"],
                "parameters": t["schema"],
            },
        }
        for t in all_tools
    ]
    
    response = litellm.completion(
        model=f"ollama/{DEFAULT_LLM_MODEL}",
        messages=messages,
        tools=tools_payload,
        api_base=OLLAMA_URL,
        temperature=0.0
    )
    
    assistant_msg = response.choices[0].message
    
    tool_calls_dict = []
    if assistant_msg.tool_calls:
        for tc in assistant_msg.tool_calls:
            import json
            tool_calls_dict.append({
                "id": tc.id,
                "name": tc.function.name,
                "args": json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
            })
            
    new_message = {"role": "assistant", "content": assistant_msg.content or ""}
    if tool_calls_dict:
        new_message["tool_calls"] = tool_calls_dict

    return {"messages": messages + [new_message], "pending_tool_call": None}

async def check_hitl_node(state: AgentState):
    """Check if the selected tool requires human-in-the-loop validation."""
    if state.get("hitl_approved") is not None:
        return {}
        
    last_msg = state["messages"][-1]
    tool_calls = last_msg.get("tool_calls", [])
    if not tool_calls:
        return {"pending_tool_call": None}
        
    tc = tool_calls[0]
    if tc["name"] in WRITE_TOOLS:
        return {
            "pending_tool_call": {
                "id": tc["id"],
                "name": tc["name"],
                "args": tc["args"]
            },
            "hitl_approved": None
        }
        
    return {"pending_tool_call": None, "hitl_approved": True}

async def execute_tools_node(state: AgentState):
    """Execute the tools (either MCP or custom python tools)."""
    last_msg = state["messages"][-1]
    tool_calls = last_msg.get("tool_calls", [])
    if not tool_calls:
        return {"messages": state["messages"]}
        
    updated_messages = list(state["messages"])
    for tc in tool_calls:
        name = tc["name"]
        args = tc["args"]
        server = tool_server_map.get(name)
        
        if server == "custom":
            try:
                result = custom_callable_map[name].invoke(args)
            except Exception as e:
                result = f"Custom tool error: {str(e)}"
        else:
            try:
                result = await execute_mcp_tool(name, args, server)
            except Exception as e:
                result = f"MCP tool error: {str(e)}"
                
        updated_messages.append({
            "role": "tool",
            "name": name,
            "tool_call_id": tc["id"],
            "content": result
        })
        
    return {
        "messages": updated_messages,
        "pending_tool_call": None,
        "hitl_approved": None
    }

async def reject_tool_node(state: AgentState):
    """Handle rejected tool call by injecting refusal feedback."""
    last_msg = state["messages"][-1]
    tool_calls = last_msg.get("tool_calls", [])
    updated_messages = list(state["messages"])
    for tc in tool_calls:
        updated_messages.append({
            "role": "tool",
            "name": tc["name"],
            "tool_call_id": tc["id"],
            "content": "Tool call rejected by user. Do not retry this action."
        })
    return {
        "messages": updated_messages,
        "pending_tool_call": None,
        "hitl_approved": None
    }

async def hitl_pause_node(state: AgentState):
    return {}

In [7]:
def route_after_agent(state: AgentState) -> Literal["check_hitl", "__end__"]:
    last_msg = state["messages"][-1]
    if last_msg.get("tool_calls"):
        return "check_hitl"
    return "__end__"

def route_after_hitl(state: AgentState) -> Literal["execute_tools", "hitl_pause", "reject_tool"]:
    approved = state.get("hitl_approved")
    pending = state.get("pending_tool_call")
    if pending and approved is None:
        return "hitl_pause"
    if approved is False:
        return "reject_tool"
    return "execute_tools"

# Compile Graph
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("check_hitl", check_hitl_node)
builder.add_node("execute_tools", execute_tools_node)
builder.add_node("reject_tool", reject_tool_node)
builder.add_node("hitl_pause", hitl_pause_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_after_agent)
builder.add_conditional_edges(
    "check_hitl",
    route_after_hitl,
    {
        "execute_tools": "execute_tools",
        "reject_tool": "reject_tool",
        "hitl_pause": "hitl_pause"
    }
)
builder.add_edge("hitl_pause", "check_hitl")
builder.add_edge("execute_tools", "agent")
builder.add_edge("reject_tool", "agent")

graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["hitl_pause"]
)
print("LangGraph Compiled Successfully!")

LangGraph Compiled Successfully!


## 4. Test Read-Only Run with Custom Tool

Let's ask the agent to compute a math question and get the local system time. This should run custom python tools and complete without hitting any HITL interrupts.

In [8]:
session_id_ro = f"litellm-ro-test-{uuid.uuid4()}"
thread_config_ro = {"configurable": {"thread_id": session_id_ro}}

input_state_ro = {
    "messages": [{"role": "user", "content": "What is 152 + 372 * 3? Also, what is the current local system time?"}],
    "pending_tool_call": None,
    "hitl_approved": None
}

print("Running graph...")
async for event in graph.astream(
    input_state_ro,
    config=thread_config_ro,
    stream_mode="values"
):
    last_msg = event["messages"][-1]
    print(f"\nMessage type: {last_msg['role']}")
    if last_msg.get("content"):
        print(f"Content: {last_msg['content']}")
    if last_msg.get("tool_calls"):
        print(f"Tool Calls: {last_msg['tool_calls']}")

snapshot = graph.get_state(thread_config_ro)
print(f"\nExecution finished! Next node: {snapshot.next}")

Running graph...

Message type: user
Content: What is 152 + 372 * 3? Also, what is the current local system time?

Message type: assistant
Content: {"function": "calculate_math", "arguments": {"expression": "152 + 372 * 3"}}

Execution finished! Next node: ()


## 5. Test Write Run with HITL Pause & Resume

Now let's ask the agent to index a document. This will trigger the `upsert_documents` write tool, causing the agent to pause. We will then resume it with approval.

In [9]:
session_id_write = f"litellm-write-test-{uuid.uuid4()}"
thread_config_write = {"configurable": {"thread_id": session_id_write}}

input_state_write = {
    "messages": [{"role": "user", "content": "Please index the document 'LiteLLM combined with LangGraph is incredibly modular' in the default collection."}],
    "pending_tool_call": None,
    "hitl_approved": None
}

print("Running write graph...")
async for event in graph.astream(
    input_state_write,
    config=thread_config_write,
    stream_mode="values"
):
    pass

snapshot = graph.get_state(thread_config_write)
print("\n--- Paused State ---")
print(f"Next node to execute: {snapshot.next}")
print(f"Pending tool call   : {snapshot.values.get('pending_tool_call')}")
print(f"HITL Approved       : {snapshot.values.get('hitl_approved')}")

Running write graph...

--- Paused State ---
Next node to execute: ('hitl_pause',)
Pending tool call   : {'id': 'call_37dd733a-3cb2-4cb5-94a9-437c1ca2e4e1', 'name': 'ingest_file_or_directory', 'args': {'path': 'LiteLLM combined with LangGraph is incredibly modular'}}
HITL Approved       : None


In [ ]:
# Grant human approval
state_update = {"hitl_approved": True, "pending_tool_call": None}
graph.update_state(thread_config_write, state_update, as_node="check_hitl")

print("Resuming graph with approval...")
async for event in graph.astream(
    None,
    config=thread_config_write,
    stream_mode="values"
):
    last_msg = event["messages"][-1]
    print(f"\nMessage type: {last_msg['role']}")
    if last_msg.get("content"):
        print(f"Content: {last_msg['content']}")

final_snapshot = graph.get_state(thread_config_write)
print(f"\nExecution finished! Next node: {final_snapshot.next}")

Resuming graph with approval...

Message type: assistant
